# Data pre-processing for LLM

## Simple token generator

In [152]:
import urllib.request
import re
url = ("https://raw.githubusercontent.com/rasbt/"
      "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
      "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x10aa61c80>)

In [154]:
with open(file_path, "r") as f:
    text = f.read()
print(text)

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through th

In [155]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[: 30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [156]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [157]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [160]:
class SimpleTokenizer1():
    def __init__(self,vocab):
        self.str_to_int = vocab
        #self.int_to_str = {token:integer for token,integer in enumerate(vocab)}
        self.int_to_str = {integer:token for token,integer in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [161]:
tokeninzer1 = SimpleTokenizer1(vocab)
text_t1 = "I HAD always thought Jack Gisburn"
ids = tokeninzer1.encode(text_t1)
c = tokeninzer1.decode(ids)
print(c)

I HAD always thought Jack Gisburn


In [162]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab2 = {token: integer for integer, token in enumerate(all_tokens)}
print(len(vocab2))

1132


In [163]:
for i, item in enumerate(list (vocab2.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [164]:
class SimpleTokenizer2():
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer:token for token,integer in vocab.items() }

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]  
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text



In [165]:
tokeninzer2 = SimpleTokenizer2(vocab2)
text_t2 = "Hello World"
ids = tokeninzer2.encode(text_t2)
d = tokeninzer2.decode(ids)
print(d)

<|unk|> <|unk|>


In [166]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text_t3 = " <|endoftext|> ".join((text1, text2))
print(text_t3)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [168]:
tokenizer2 = SimpleTokenizer2(vocab2)
print(tokenizer2.encode(text_t3))
print(tokenizer2.decode(tokenizer2.encode(text_t3)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte Pair encoding using tiktoken

In [169]:
%pip uninstall -y tiktoken

Found existing installation: tiktoken 0.14.0
Uninstalling tiktoken-0.14.0:
  Successfully uninstalled tiktoken-0.14.0
Note: you may need to restart the kernel to use updated packages.


In [170]:
%pip install tiktoken

  Using cached tiktoken-0.14.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.7 kB)
Using cached tiktoken-0.14.0-cp314-cp314-macosx_11_0_arm64.whl (1.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [144]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


In [145]:
tokenizer = tiktoken.get_encoding("gpt2")
text_t4 =("Akwirw ier.")
integers = tokenizer.encode(text_t4, allowed_special={"<|endoftext|>"})
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959, 13]
Akwirw ier.


## Data Sliding with simple window

In [179]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})
print(len(enc_text))

5145


In [180]:
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:  {y}")

x: [290, 4920, 2241, 287]
y:  [4920, 2241, 287, 257]


In [182]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [181]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
    

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [183]:
print(tokenizer.decode(enc_sample[0:50]))

 and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Th


## 

In [124]:
pip install torch

  Using cached torch-2.14.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (39 kB)
Using cached torch-2.14.1-cp314-cp314-macosx_14_0_arm64.whl (127.3 MB)
Note: you may need to restart the kernel to use updated packages.


In [151]:
print(torch.backends.mps.is_available())

True


### A dataset for batched inputs and targets

In [223]:
import torch
from torch.utils.data import Dataset, DataLoader   

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx])

### A data loader to generate batches with input-with pairs

In [239]:
def create_dataloaders(txt, batch_size = 4, max_length = 256, stride = 128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [240]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataLoader = create_dataloaders(raw_text, batch_size=2, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataLoader)
first_batch = next(data_iter)
print(first_batch)


[tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])]


/var/folders/0w/tc2y97x135b_c16y23ndpjrh0000gn/T/ipykernel_23242/2829990190.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx])


In [238]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632]]), tensor([[ 2138,   257,  7026, 15632],
        [  257,  7026, 15632,   438]])]


/var/folders/0w/tc2y97x135b_c16y23ndpjrh0000gn/T/ipykernel_23242/2829990190.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx])


In [242]:
data_loader = create_dataloaders(raw_text, batch_size=2, max_length=4, stride=5, shuffle=True)
data_iter = iter(data_loader)
input_batch, target_batch = next(data_iter)
print(input_batch)
print(target_batch)

tensor([[ 3960,   503,    25,   366],
        [ 1544, 45111,  2241,   866]])
tensor([[  503,    25,   366,  3856],
        [45111,  2241,   866,   287]])


/var/folders/0w/tc2y97x135b_c16y23ndpjrh0000gn/T/ipykernel_23242/2829990190.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx])


In [246]:
input_ids = torch.tensor([2, 3, 5, 1])
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(num_embeddings=6, embedding_dim=4)
print(embedding_layer(input_ids))
print(embedding_layer(torch.tensor(2)))

tensor([[-0.1606, -0.4015,  0.6957, -1.8061],
        [ 1.8960, -0.1750,  1.3689, -1.6033],
        [ 0.9985,  0.2212,  1.8319, -0.3378],
        [ 0.3486,  0.6603, -0.2196, -0.3792]], grad_fn=<EmbeddingBackward0>)
tensor([-0.1606, -0.4015,  0.6957, -1.8061], grad_fn=<EmbeddingBackward0>)


In [262]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
dataLoader = create_dataloaders(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataLoader)
inputs, target = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


/var/folders/0w/tc2y97x135b_c16y23ndpjrh0000gn/T/ipykernel_23242/2829990190.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.input_ids[idx]), torch.tensor(self.target_ids[idx])


In [263]:
token_embedding = token_embedding_layer(inputs)
print("Token Embeddings shape:\n", token_embedding.shape)

Token Embeddings shape:
 torch.Size([8, 4, 256])


In [266]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional Embeddings shape:\n", pos_embeddings.shape)

Positional Embeddings shape:
 torch.Size([4, 256])


In [267]:
inputs_with_pos = token_embedding + pos_embeddings
print("Inputs with Positional Embeddings shape:\n", inputs_with_pos.shape)

Inputs with Positional Embeddings shape:
 torch.Size([8, 4, 256])
